# Лабораторная работа № 1

## Однородные координаты и группа SE(3). Дерево систем координат

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 2. Аффинная геометрия сцены. Однородные координаты и группа SE(3)

---

**Студент:** _фамилия, имя, группа_
**Дата:**

---

### Цель работы

Реализовать жёсткое преобразование трёхмерного пространства как элемент группы $SE(3)$,
проверить свойства группы численным экспериментом, построить дерево систем координат
и оценить накопление вычислительной погрешности при многократной композиции.

### Что понадобится

`numpy`, `matplotlib`. Ничего больше.

### Правило практикума

Алгоритм вы реализуете сами. Готовые функции из библиотек пространственных преобразований
(`scipy.spatial.transform`, `spatialmath`, `pytransform3d`) допустимо использовать **только**
для сверки собственного результата — и такую сверку нужно показать явно.

## Теоретическая справка

**Жёсткое преобразование** переводит точку $p \in \mathbb{R}^3$ в точку $Rp + t$, где
$R$ — матрица поворота, $t$ — вектор сдвига. Такое отображение не является линейным
(из-за сдвига), поэтому одной матрицей $3\times 3$ его не записать.

**Однородные координаты** решают эту проблему: точку $p = (x, y, z)$ представляют
четвёркой $(x, y, z, 1)$, а вектор (направление, разность точек) — четвёркой $(x, y, z, 0)$.
Тогда преобразование становится линейным и записывается матрицей $4\times4$:

$$
T \;=\;
\begin{pmatrix} R & t \\ 0^{\mathsf T} & 1 \end{pmatrix},
\qquad
R \in SO(3): \quad R^{\mathsf T} R = I, \;\; \det R = +1, \qquad t \in \mathbb{R}^3 .
$$

Множество таких матриц образует **группу** $SE(3)$ относительно умножения:

* композиция двух преобразований снова является преобразованием (замкнутость);
* умножение ассоциативно: $(AB)C = A(BC)$;
* нейтральный элемент — единичная матрица $I$;
* обратный элемент существует и вычисляется в замкнутой форме:

$$
T^{-1} = \begin{pmatrix} R^{\mathsf T} & -R^{\mathsf T} t \\ 0^{\mathsf T} & 1 \end{pmatrix}.
$$

Обратите внимание: обращать матрицу численно (`np.linalg.inv`) не нужно и не следует —
формула выше даёт точный результат за $O(1)$ операций.

Группа **некоммутативна**: $AB \ne BA$. Это не техническая деталь, а физический факт —
повернуть, а затем сдвинуть, и сдвинуть, а затем повернуть, приводит в разные точки.

**Дерево систем координат.** У робота каждое звено несёт свою систему координат, заданную
относительно родительской. Совокупность образует дерево с корнем в мировом кадре.
Преобразование между двумя произвольными кадрами получается перемножением матриц вдоль
пути через общего предка. Именно это делает библиотека TF2 в ROS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## Задание 1. Элементарные повороты и класс преобразования

Реализуйте три функции элементарных поворотов и класс `Transform`.

**Требования к классу:**

* хранит матрицу $4\times4$ в поле `M`;
* конструктор `from_Rt(R, t)`; по умолчанию — тождественное преобразование;
* свойства `R` и `t`;
* композиция через оператор `@` (метод `__matmul__`), возвращающая новый `Transform`;
* `inverse()` — **по формуле**, без `np.linalg.inv`;
* `apply_point(p)` и `apply_vector(v)`, работающие и с одной точкой, и с массивом точек $N\times3$.

*Подсказка по знакам:* поворот вокруг оси $z$ на угол $\alpha$ переводит орт $e_x$ в
$(\cos\alpha, \sin\alpha, 0)$.

In [ ]:
def rotx(a):
    # TODO: матрица поворота вокруг оси X на угол a
    raise NotImplementedError


def roty(a):
    # TODO
    raise NotImplementedError


def rotz(a):
    # TODO
    raise NotImplementedError


class Transform:
    """Жёсткое преобразование как элемент SE(3), хранимое матрицей 4x4."""

    def __init__(self, matrix=None):
        self.M = np.eye(4) if matrix is None else np.asarray(matrix, dtype=float)
        if self.M.shape != (4, 4):
            raise ValueError('нужна матрица 4x4, получено %s' % (self.M.shape,))

    @classmethod
    def from_Rt(cls, R, t):
        # TODO: собрать матрицу 4x4 из R и t
        raise NotImplementedError

    @property
    def R(self):
        return self.M[:3, :3]

    @property
    def t(self):
        return self.M[:3, 3]

    def __matmul__(self, other):
        # TODO: композиция преобразований
        raise NotImplementedError

    def inverse(self):
        # TODO: по формуле обращения, БЕЗ np.linalg.inv
        raise NotImplementedError

    def apply_point(self, p):
        # TODO: однородная координата w = 1
        raise NotImplementedError

    def apply_vector(self, v):
        # TODO: однородная координата w = 0
        raise NotImplementedError

    def __repr__(self):
        return 'Transform(\n%s)' % np.array2string(self.M, precision=4, suppress_small=True)

**Проверка.** Ячейка ниже должна отработать без ошибок.

In [ ]:
T = Transform.from_Rt(rotz(np.pi / 6) @ rotx(np.pi / 4), [1.0, -2.0, 0.5])
resid = np.linalg.norm((T @ T.inverse()).M - np.eye(4))
print('||T @ T^-1 - I|| = %.3e' % resid)
assert resid < 1e-12, 'обращение реализовано неверно'
print('OK')

## Задание 2. Точка и вектор — в чём разница

Возьмите $p = (1, 0, 0)$ и примените к нему `apply_point` и `apply_vector`.

**Что показать в отчёте:** численно подтвердить, что сдвиг действует на точку и **не действует**
на вектор, то есть `apply_vector(p)` совпадает с $Rp$ с точностью до машинной погрешности.

**Вопрос для вывода:** какие физические величины следует преобразовывать как векторы,
а не как точки? Приведите два примера.

In [ ]:
# TODO: задание 2

## Задание 3. Свойства группы — численная проверка

Напишите функцию `random_transform(rng)`, порождающую случайный элемент $SE(3)$
(случайные углы Эйлера и случайный сдвиг). Возьмите три случайных элемента $A$, $B$, $C$ и проверьте:

1. **ассоциативность:** $\|(AB)C - A(BC)\|$ — должно быть порядка $10^{-15}$;
2. **некоммутативность:** $\|AB - BA\|$ — должно быть **не** малым; приведите конкретный контрпример;
3. **замкнутость:** для произведения $AB$ проверьте $\|R^{\mathsf T}R - I\|$ и $\det R$.

**Вопрос для вывода:** почему проверка ассоциативности даёт не ноль, а $10^{-15}$?
Что именно измеряет эта величина?

In [ ]:
# TODO: задание 3

## Задание 4. Дерево систем координат

Ниже задано дерево кадров робота с камерой на схвате и столом с объектом:

```
world ─┬─ base ── shoulder ── elbow ── wrist ── camera
       └─ table ── object
```

Реализуйте три функции:

* `chain_to_root(frame)` — список кадров от заданного до корня;
* `world_from(frame)` — преобразование из кадра `frame` в мировой;
* `lookup(target, source)` — преобразование, переводящее координаты **из** `source` **в** `target`.

Это в точности то, что делает `tf2` в ROS: `lookup_transform(target, source)`.

**Что вычислить:** положение начала кадра `object` в системе координат `camera`.

**Две проверки, которые обязаны пройти:**

1. $\mathrm{lookup}(a,b)\cdot\mathrm{lookup}(b,a) = I$;
2. результат, полученный через `lookup`, совпадает с результатом пересчёта «вручную» через мировой кадр.

In [ ]:
PARENT = {'base': 'world', 'table': 'world', 'shoulder': 'base', 'elbow': 'shoulder',
          'wrist': 'elbow', 'camera': 'wrist', 'object': 'table'}

LOCAL = {
    'base':     Transform.from_Rt(rotz(0.30), [0.00, 0.00, 0.20]),
    'table':    Transform.from_Rt(np.eye(3),  [1.20, 0.40, 0.00]),
    'shoulder': Transform.from_Rt(roty(0.45), [0.00, 0.00, 0.35]),
    'elbow':    Transform.from_Rt(roty(-0.80), [0.40, 0.00, 0.00]),
    'wrist':    Transform.from_Rt(rotx(0.60), [0.35, 0.00, 0.00]),
    'camera':   Transform.from_Rt(rotz(-np.pi / 2) @ roty(np.pi / 2), [0.05, 0.00, 0.10]),
    'object':   Transform.from_Rt(rotz(1.10), [0.10, -0.15, 0.75]),
}


def chain_to_root(frame):
    # TODO
    raise NotImplementedError


def world_from(frame):
    # TODO
    raise NotImplementedError


def lookup(target, source):
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: вычислить положение object в кадре camera и выполнить обе проверки

## Задание 5. Визуализация дерева координат

Постройте трёхмерный рисунок: для каждого кадра нарисуйте координатный триэдр
(три отрезка длиной ~0,18 из начала кадра вдоль его осей, разными цветами) и подпишите имя кадра.

Оси стройте **через `apply_point`** — это заодно проверяет корректность класса.

**Что показать в отчёте:** рисунок и словесное объяснение, почему кадр `camera` повёрнут
относительно `wrist` именно так.

In [ ]:
# TODO: задание 5

## Задание 6. Накопление погрешности

Возьмите преобразование малого шага `Tstep` (поворот порядка $10^{-3}$ рад и сдвиг порядка $10^{-3}$).
Для набора значений $N$ (логарифмическая сетка от $10$ до $10^5$):

1. умножьте тождественное преобразование на `Tstep` ровно $N$ раз;
2. затем умножьте результат на `Tstep.inverse()` тоже $N$ раз;
3. теоретически должна получиться единичная матрица — измерьте $\|T_{\text{acc}} - I\|$;
4. отдельно измерьте потерю ортогональности $\|R^{\mathsf T}R - I\|$.

Постройте оба графика в двойном логарифмическом масштабе и оцените наклон
(`np.polyfit` по логарифмам).

**Вопросы для вывода:**

* Какому наклону соответствует накопление погрешности, пропорциональное $N$? А пропорциональное $\sqrt N$?
* Какой наклон получился у вас и что это означает: ошибки накапливаются независимо или систематически?
* Оцените результат в единицах машинного эпсилон (`np.finfo(float).eps`). Опасна ли такая
  погрешность для робота с рабочей зоной в 1 метр?

In [ ]:
# TODO: задание 6

## Задание 7 (дополнительное). Ортогонализация через SVD

При длительных вычислениях матрица поворота «портится»: $R^{\mathsf T}R$ перестаёт быть единичной.
Ближайшая в норме Фробениуса ортогональная матрица находится через сингулярное разложение
$R = U\Sigma V^{\mathsf T}$:

$$R_{\text{испр}} = U \,\mathrm{diag}(1, 1, \det(UV^{\mathsf T}))\, V^{\mathsf T}.$$

Множитель с определителем нужен, чтобы гарантированно получить поворот, а не отражение.

**Что сделать:** взять заведомо известную $R_{\text{ист}}$, добавить шум $\sigma \approx 2\cdot10^{-3}$,
восстановить и сравнить: ошибка относительно $R_{\text{ист}}$ и неортогональность — до и после.

Этот приём — задел к теме 3, где сингулярное разложение появится уже как основной инструмент.

In [ ]:
# TODO: задание 7

## Выводы

_Напишите здесь содержательные выводы. Требуется ответить на все вопросы, отмеченные
в заданиях как «вопрос для вывода», и указать численные значения, которые вы получили._

## Контрольные вопросы

1. Почему жёсткое преобразование нельзя записать матрицей $3\times3$?
2. Чем отличается однородное представление точки от однородного представления вектора?
3. Выведите формулу обращения $T^{-1}$. Почему она предпочтительнее `np.linalg.inv`?
4. Приведите геометрический контрпример к коммутативности $SE(3)$.
5. Сколько независимых параметров у элемента $SE(3)$? А у матрицы $4\times4$? Куда делись остальные?
6. Как по дереву кадров вычислить преобразование между двумя ветвями?
7. Что измеряет величина $\|R^{\mathsf T}R - I\|$ и почему она растёт с числом операций?